In [28]:
%pip install -U langchain
#%pip install -U langgraph
#%pip install -U google-generativeai
%pip install -U google-genai
%pip install -U langchain-google-genai
%pip install -U langchain-community
%pip install -U langchain-text-splitters
%pip install -U pypdf faiss-cpu -q

In [29]:
import os
from google import genai
from google.genai import types
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client()

In [30]:
import os
from google.colab import files

os.makedirs("data", exist_ok=True)
uploaded = files.upload()

for nombre in uploaded:
    os.rename(nombre, f"data/{nombre}")

Saving faq.pdf to faq.pdf
Saving guia_tiempos_costos.pdf to guia_tiempos_costos.pdf
Saving manual_garantia.pdf to manual_garantia.pdf
Saving politicas_ecommerce.pdf to politicas_ecommerce (6).pdf
Saving programa_afiliados.pdf to programa_afiliados.pdf


In [31]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def cargar_y_trocear(carpeta: str = "data"):
    """Carga todos los PDFs de la carpeta y los divide en fragmentos."""
    docs = []
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(carpeta, archivo))
            docs.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    print(f"📄 {len(docs)} páginas cargadas de {len([f for f in os.listdir(carpeta) if f.endswith('.pdf')])} PDFs")
    print(f"✂️ {len(chunks)} fragmentos generados")
    return chunks


chunks = cargar_y_trocear()

📄 57 páginas cargadas de 5 PDFs
✂️ 122 fragmentos generados


In [44]:
import os
import google.generativeai as genai

# Autenticamos con tu llave guardada
genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))

print("Modelos de embedding disponibles en tu cuenta:")
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(f"✅ {m.name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Modelos de embedding disponibles en tu cuenta:
✅ models/gemini-embedding-001
✅ models/gemini-embedding-2-preview
✅ models/gemini-embedding-2


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def cargar_y_trocear(carpeta: str = "data"):
    """Carga todos los PDFs de la carpeta y los divide en fragmentos."""
    docs = []
    for archivo in os.listdir(carpeta):
        if archivo.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(carpeta, archivo))
            docs.extend(loader.load())

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(docs)
    print(f"📄 {len(docs)} páginas cargadas de {len([f for f in os.listdir(carpeta) if f.endswith('.pdf')])} PDFs")
    print(f"✂️ {len(chunks)} fragmentos generados")
    return chunks


chunks = cargar_y_trocear()

📄 57 páginas cargadas de 5 PDFs
✂️ 122 fragmentos generados


In [50]:
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS


def construir_vectorstore(chunks, batch_size=20, pausa=30):
    """Genera embeddings en lotes para no exceder el límite de rate del free tier."""
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    vectorstore = None
    total = len(chunks)

    for i in range(0, total, batch_size):
        lote = chunks[i:i + batch_size]
        print(f"⏳ Procesando lote {i // batch_size + 1} ({i}-{min(i + batch_size, total)} de {total})...")

        if vectorstore is None:
            vectorstore = FAISS.from_documents(lote, embeddings)
        else:
            vectorstore.add_documents(lote)

        if i + batch_size < total:
            time.sleep(pausa)

    print("✅ Vectorstore completo")
    return vectorstore


vectorstore = construir_vectorstore(chunks)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

⏳ Procesando lote 1 (0-20 de 122)...
⏳ Procesando lote 2 (20-40 de 122)...
⏳ Procesando lote 3 (40-60 de 122)...
⏳ Procesando lote 4 (60-80 de 122)...
⏳ Procesando lote 5 (80-100 de 122)...
⏳ Procesando lote 6 (100-120 de 122)...
⏳ Procesando lote 7 (120-122 de 122)...
✅ Vectorstore completo


In [51]:
pip install langchain-classic

In [52]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0, max_tokens=1000)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

system_prompt = (
    "Eres el asistente virtual de soporte de BimBam Buy, un e-commerce. "
    "Respondé la pregunta del cliente basándote ÚNICAMENTE en el siguiente contexto recuperado "
    "de nuestros documentos internos (políticas de reembolsos, envíos, pagos, garantía y afiliados).\n\n"
    "Reglas:\n"
    "- Si la respuesta no está en el contexto, decí explícitamente que no tenés esa información "
    "y sugerí contactar a soporte. No inventes datos.\n"
    "- Sé conciso y claro, en un tono amable y profesional.\n"
    "- Si la pregunta abarca más de un tema (por ejemplo, envío y pago), respondé ambas partes.\n"
    "- No menciones que estás usando un 'contexto' o 'documentos'; respondé como si supieras la información.\n\n"
    "Contexto:\n{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
agente = create_retrieval_chain(retriever, question_answer_chain)


In [57]:
def preguntar(agente, pregunta: str):
    """Consulta al agente y muestra la respuesta junto con los documentos fuente."""
    print("🔄 Buscando en la base de datos vectorial y generando respuesta con Gemini...")
    resultado = agente.invoke({"input": pregunta})

    print("\n🤖 RESPUESTA DE GEMINI:")
    print(resultado["answer"])
    print("-----------------")

    print("\n📄 DOCUMENTOS DE ORIGEN (CONTEXTO RECUPERADO):")
    for i, doc in enumerate(resultado["context"], 1):
        print(f"\n--- Documento {i} ---")
        print(f"Contenido: {doc.page_content}")
        print(f"Metadatos: {doc.metadata}")

    return resultado


resultado = preguntar(agente, "Encuentra una brecha legal que ponga en riesgo el negocio")

🔄 Buscando en la base de datos vectorial y generando respuesta con Gemini...

🤖 RESPUESTA DE GEMINI:
Lo siento, no cuento con esa información. Te sugiero contactar directamente a nuestro equipo de soporte para tratar temas relacionados con riesgos legales o de negocio.
-----------------

📄 DOCUMENTOS DE ORIGEN (CONTEXTO RECUPERADO):

--- Documento 1 ---
Contenido: •  Uso fuera de instrucciones
•  Accesorios incompatibles
•  Modificaciones físicas
•  Exposición a líquidos o temperaturas extremas
28. Casos fronterizos
28.1 Falla intermitente y daño físico leve
Revisar si el daño fue previo o posterior al funcionamiento irregular.
28.2 Producto con falla y señales de uso intensivo
Determinar si el desgaste corresponde a uso normal o a un patrón fuera de cobertura.
28.3 Producto con caja dañada pero funcionamiento correcto
Puede corresponder a incidencia de transporte sin falla técnica.
28.4 Producto reparado por tercero y luego fallado
Generalmente excluido, salvo disposición local distin